In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time

# Data Reading

In [ ]:
df = spark.read.format("delta")\
    .load("s3://learn-databricks-project-e2e-1-bronze/customers")
    
df.show()

In [ ]:
df = df.drop("_rescued_data")
df.show()

In [ ]:
df = df.withColumn("domains", split(col("email"), "@")[1])
df.show()

In [ ]:
df2 = df.groupBy("domains").agg(count("customer_id").alias("total_customers"))\
    .sort("total_customers", ascending=False)
df2.show()

In [ ]:
df_gmail = df.filter(col("domains") == "gmail.com")
df_gmail.show()
time.sleep(5)  # Simulate a delay for demonstration purposes

df_yahoo = df.filter(col("domains") == "yahoo.com")
df_yahoo.show()
time.sleep(5) 

df_hotmail = df.filter(col("domains") == "hotmail.com")
df_hotmail.show()
time.sleep(5) 

In [ ]:
df_wfn = df.withColumn("full_name", concat(col('first_name'), lit(' '), col('last_name')))\
    .drop("first_name", "last_name")
    
df_wfn.show()

In [ ]:
df_wfn.write.format("delta")\
    .mode("overwrite")\
    .save("s3://learn-databricks-project-e2e-1-silver/customers")